# Pandas 专项训练：缺失值识别与标准化

## 练习主题

识别并统一业务数据中的真实缺失值与伪装缺失值。

## 业务背景

某设备运维系统导出了一批巡检记录。

由于数据来自人工录入和不同系统，部分字段存在：

- 前后空格
- 大小写不统一
- 空字符串
- `N/A`
- `NULL`
- `-`
- `unknown`
- Python 原生缺失值

这些值必须先统一处理，才能正确统计缺失情况。

## 训练目标

1. 保留原始数据，创建清洗副本。
2. 检查清洗前 Pandas 能直接识别的缺失值。
3. 清除文本字段前后的空格。
4. 统一设备编号、站点编号和状态字段的大小写。
5. 将伪装缺失值统一转换为 `pd.NA`。
6. 生成字段级缺失情况报告。
7. 根据字段业务含义分别使用删除、填充或保留策略。
8. 验证清洗结果是否符合要求。

## 清洗规则

### 文本格式标准化

- `device_id`：清除空格并转换为大写。
- `site`：清除空格并转换为大写。
- `status`：清除空格并转换为大写。
- `signal_strength`：只清除前后空格，暂时不转换数值类型。
- `technician`：只清除前后空格。

### 缺失值标准化

下列内容均视为缺失值：

- 空字符串
- 只有空格的字符串
- `N/A`
- `NULL`
- `-`
- `UNKNOWN`

将它们统一转换为 `pd.NA`。

### 业务处理规则

- 删除 `device_id` 缺失的记录。
- `status` 缺失时填充为 `UNKNOWN`。
- `technician` 缺失时填充为 `UNASSIGNED`。
- `signal_strength` 缺失值暂时保留，不删除、不填充。

## 限制条件

- 不允许手工逐行修改数据。
- 不允许直接修改原始 DataFrame。
- 不允许对整个 DataFrame 直接执行 `dropna()`。
- 必须使用 Pandas 向量化方法完成清洗。

In [1]:
import pandas as pd

data = {
    "record_id": range(1, 13),

    "device_id": [
        "r34-01",
        "r34-01",
        "r34-02",
        "r34-02",
        " r34-03 ",
        "r34-03",
        None,
        "r34-04",
        "r34-04",
        "r34-05",
        "r34-05",
        "r34-06"
    ],

    "site": [
        "R34",
        " R34 ",
        "r34",
        "R34",
        "R35",
        "R35",
        "R36",
        " R36 ",
        "R36",
        "R37",
        "r37",
        "R38"
    ],

    "status": [
        "NORMAL",
        " normal ",
        "ERROR",
        " ",
        "N/A",
        None,
        "NORMAL",
        "NULL",
        "ERROR",
        "-",
        "normal",
        " ERROR "
    ],

    "signal_strength": [
        "18.5",
        "19.1",
        " ",
        "17.8",
        "unknown",
        None,
        "20.0",
        "21.2",
        "NULL",
        "19.9",
        "-",
        "18.7"
    ],

    "technician": [
        "Li",
        " Li ",
        "Wang",
        "Wang",
        "",
        "N/A",
        "Chen",
        "Chen",
        "NULL",
        "Zhao",
        "zhao",
        "-"
    ],

    "inspect_time": [
        "2026-07-01 08:00",
        "2026-07-01 08:10",
        "2026-07-01 08:20",
        "2026-07-01 08:30",
        "2026-07-01 08:40",
        "2026-07-01 08:50",
        "2026-07-01 09:00",
        "2026-07-01 09:10",
        "2026-07-01 09:20",
        "2026-07-01 09:30",
        "2026-07-01 09:40",
        "2026-07-01 09:50"
    ]
}

df_raw = pd.DataFrame(data)

df_raw

,record_id,device_id,site,status,signal_strength,technician,inspect_time
0,1,r34-01,R34,NORMAL,18.5,Li,2026-07-01 08:00
1,2,r34-01,R34,normal,19.1,Li,2026-07-01 08:10
2,3,r34-02,r34,ERROR,,Wang,2026-07-01 08:20
3,4,r34-02,R34,,17.8,Wang,2026-07-01 08:30
4,5,r34-03,R35,N/A,unknown,,2026-07-01 08:40
5,6,r34-03,R35,NaN,NaN,N/A,2026-07-01 08:50
6,7,NaN,R36,NORMAL,20.0,Chen,2026-07-01 09:00
7,8,r34-04,R36,NULL,21.2,Chen,2026-07-01 09:10
8,9,r34-04,R36,ERROR,NULL,NULL,2026-07-01 09:20
9,10,r34-05,R37,-,19.9,Zhao,2026-07-01 09:30


### 一、检查数据原始表达

In [14]:
# 创建原始字段检查表

check_cols = [
    'device_id',
    'site',
    'status',
    'signal_strength',
    'technician',
    'inspect_time'
]

In [ ]:
# 1. # 检查数据规模，为清洗前后的行列数量对比建立基准
df_raw.shape

(12, 7)

In [ ]:
# 2. 检查字段类型、非空数量和数据整体结构
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   record_id        12 non-null     int64
 1   device_id        11 non-null     str  
 2   site             12 non-null     str  
 3   status           11 non-null     str  
 4   signal_strength  11 non-null     str  
 5   technician       12 non-null     str  
 6   inspect_time     12 non-null     str  
dtypes: int64(1), str(6)
memory usage: 1.2 KB


In [ ]:
# 3. 统计 Pandas 当前已经识别的真实缺失值
df_raw.isna().sum()

record_id          0
device_id          1
site               0
status             1
signal_strength    1
technician         0
inspect_time       0
dtype: int64

In [ ]:
# 4.检查重点字段的原始表达方式及其出现次数
for col in check_cols:
    print(f"\n===== {col} =====")
    print(df_raw[col].map(repr).value_counts())



===== device_id =====
device_id
'r34-01'      2
'r34-02'      2
'r34-04'      2
'r34-05'      2
' r34-03 '    1
'r34-03'      1
nan           1
'r34-06'      1
Name: count, dtype: int64

===== site =====
site
'R34'      2
'R35'      2
'R36'      2
' R34 '    1
'r34'      1
' R36 '    1
'R37'      1
'r37'      1
'R38'      1
Name: count, dtype: int64

===== status =====
status
'NORMAL'      2
'ERROR'       2
' normal '    1
' '           1
'N/A'         1
nan           1
'NULL'        1
'-'           1
'normal'      1
' ERROR '     1
Name: count, dtype: int64

===== signal_strength =====
signal_strength
'18.5'       1
'19.1'       1
' '          1
'17.8'       1
'unknown'    1
nan          1
'20.0'       1
'21.2'       1
'NULL'       1
'19.9'       1
'-'          1
'18.7'       1
Name: count, dtype: int64

===== technician =====
technician
'Wang'    2
'Chen'    2
'Li'      1
' Li '    1
''        1
'N/A'     1
'NULL'    1
'Zhao'    1
'zhao'    1
'-'       1
Name: count, dtype: int64

=